In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# MS036 Hippocampal Ripple / IED Detection

Single-subject notebook for hippocampal ripple detection during `feedback_start` and `cue_on`, with same-electrode / same-trial IED exclusion and ripple-locked summary plots.

- Subject: `MS036`
- Condition: `Day1`
- No disk outputs are written in this pass.
- Detector logic stays notebook-local but is wrapped in a reusable `run_subject_ripple_pipeline(...)` function.

In [ ]:
import os
import sys
import math
import tempfile
import warnings
from pathlib import Path

os.environ.setdefault("NUMBA_DISABLE_JIT", "1")
mpl_dir = Path(tempfile.gettempdir()) / "memorybandit_ripple_mpl"
mpl_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_dir))
os.environ.setdefault("XDG_CACHE_HOME", str(mpl_dir))

import numpy as np
import pandas as pd
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import mne

from IPython.display import display
from joblib import Parallel, delayed
from mne.filter import next_fast_len
from scipy.signal import firwin, filtfilt, find_peaks

warnings.filterwarnings("ignore")
sns.set_context("talk")
sns.set_style("white")

def resolve_lfpanalysis():
    try:
        import LFPAnalysis  # type: ignore
        return LFPAnalysis
    except ImportError:
        repo_root = Path("/Users/salmanqasim/Documents/GitRepos/LFPAnalysis")
        if not repo_root.exists():
            raise
        if str(repo_root) not in sys.path:
            sys.path.append(str(repo_root))
        import LFPAnalysis  # type: ignore
        return LFPAnalysis

LFPAnalysis = resolve_lfpanalysis()
from LFPAnalysis.oscillation_utils import hilbert as lfp_hilbert

from membandit.io import Subject

BASE_DIR = Path("/Volumes/T7 Shield")
LOAD_DIR = BASE_DIR / "work" / "qasims01" / "MemoryBanditData" / "EMU"
EPHYS_ANALYSES_DIR = BASE_DIR / "projects" / "guLab" / "Salman" / "EphysAnalyses"
SUBJECT_ID = "MS036"
CONDITION = "Day1"

EVENT_SPECS = {
    "feedback_start": {
        "window": (0.0, 1.5),
        "trial_column": "trials_gamble",
    },
    "cue_on": {
        "window": (0.0, 1.0),
        "trial_column": "trials_mem",
    },
}

IED_BAND = (25.0, 60.0)
RIPPLE_DETECTION_BAND = (70.0, 180.0)
RIPPLE_TRACE_BAND = (70.0, 180.0)
SMOOTH_CUTOFF_HZ = 40.0
IED_Z_THRESHOLD = 4.0
RIPPLE_UPPER_SD = 4.0
RIPPLE_LOWER_SD = 2.0
RIPPLE_CLIP_SD = 4.0
RIPPLE_MERGE_GAP_S = 0.030
RIPPLE_MIN_DURATION_S = 0.020
RIPPLE_MAX_DURATION_S = 0.200
IED_EXCLUSION_WINDOW_S = 0.100
MIN_VALID_TRIALS = 10
EXAMPLE_CONTEXT_S = 0.20
SUMMARY_WINDOW_S = 0.25
SUMMARY_COMPUTE_WINDOW_S = 1.0
TFR_BASELINE_WINDOW = (-0.75, -0.10)
MIN_ELECTRODE_SUMMARY_RIPPLES = 20
UV_SCALE = 1e6
TOP_N_EXAMPLES = 6

IED_COLUMNS = [
    "participant", "event", "trial_number", "trial_index_zero_based", "electrode", "electrode_region",
    "electrode_hemi", "ied_peak_s", "ied_peak_sample", "ied_peak_z"
]
RIPPLE_CANDIDATE_COLUMNS = [
    "participant", "event", "condition", "trial_column", "trial_number", "trial_index_zero_based", "electrode",
    "electrode_region", "electrode_hemi", "ripple_start_s", "ripple_end_s", "ripple_peak_s", "ripple_duration_s",
    "ripple_start_sample", "ripple_end_sample", "ripple_peak_sample", "peak_smoothed_power", "nearest_ied_delta_s",
    "ied_within_100ms", "retained_after_ied_exclusion", "rejection_reason"
]
RIPPLE_EVENT_COLUMNS = [
    "participant", "event", "condition", "trial_column", "trial_number", "trial_index_zero_based", "electrode",
    "electrode_region", "electrode_hemi", "ripple_start_s", "ripple_end_s", "ripple_peak_s", "ripple_duration_s",
    "ripple_start_sample", "ripple_end_sample", "ripple_peak_sample", "peak_smoothed_power", "nearest_ied_delta_s",
    "ied_within_100ms"
]
TRIAL_RATE_COLUMNS = [
    "participant", "event", "condition", "trial_number", "trial_index_zero_based", "trial_column", "n_ripples",
    "n_ieds", "n_hpc_electrodes", "trial_window_s", "ripple_rate_hz"
]
ELECTRODE_RATE_COLUMNS = [
    "participant", "event", "condition", "electrode", "electrode_region", "electrode_hemi", "n_trials", "n_ripples",
    "n_ieds", "mean_ripples_per_trial", "mean_ripple_rate_hz"
]
QC_COLUMNS = [
    "participant", "event", "status", "reason", "n_hpc_electrodes", "n_valid_trials"
]

print("LFPAnalysis:", LFPAnalysis.__file__)
print("Base dir exists:", BASE_DIR.exists())
print("Load dir exists:", LOAD_DIR.exists())
print("Ephys analyses dir exists:", EPHYS_ANALYSES_DIR.exists())
print("Subject:", SUBJECT_ID)


In [ ]:
def _ensure_odd(value):
    value = int(value)
    return value if value % 2 == 1 else value + 1


def _empty_df(columns):
    return pd.DataFrame(columns=list(columns))


def bandpass_filter(data, sfreq, band, fir_window="hamming"):
    data = np.asarray(data, dtype=np.float64)
    return mne.filter.filter_data(
        data,
        sfreq=sfreq,
        l_freq=float(band[0]),
        h_freq=float(band[1]),
        method="fir",
        phase="zero-double",
        fir_window=fir_window,
        verbose=False,
    )


def compute_fast_envelope(data, sfreq, band, fir_window="hamming"):
    filtered = bandpass_filter(data, sfreq=sfreq, band=band, fir_window=fir_window)
    n_times = filtered.shape[-1]
    n_fft = next_fast_len(n_times)
    analytic = lfp_hilbert(filtered, N=n_fft, axis=-1)[..., :n_times]
    envelope = np.abs(analytic)
    return filtered, analytic, envelope


def lms_loc_scale(x):
    x = np.asarray(x, dtype=np.float64).ravel()
    x = x[np.isfinite(x)]
    if x.size < 2:
        raise ValueError("Need at least two finite samples for robust location / scale.")
    location = float(np.median(x))
    mad = float(np.median(np.abs(x - location)))
    scale = 1.4826 * mad
    if not np.isfinite(scale) or scale <= 0:
        scale = float(np.std(x, ddof=0))
    if not np.isfinite(scale) or scale <= 0:
        scale = 1e-12
    return location, scale


def smooth_power_kaiser(data, sfreq, cutoff_hz=40.0, beta=8.6, numtaps=None):
    data = np.asarray(data, dtype=np.float64)
    if cutoff_hz >= (sfreq / 2.0):
        raise ValueError("Low-pass cutoff must be below Nyquist.")
    if numtaps is None:
        numtaps = _ensure_odd(max(31, math.ceil(5 * sfreq / cutoff_hz)))
    taps = firwin(
        numtaps,
        cutoff=float(cutoff_hz),
        fs=float(sfreq),
        window=("kaiser", float(beta)),
        pass_zero="lowpass",
    )
    if data.shape[-1] <= 3:
        return data.copy(), taps
    padlen = min(data.shape[-1] - 1, 3 * (len(taps) - 1))
    smoothed = filtfilt(taps, [1.0], data, axis=-1, padlen=padlen)
    return smoothed, taps


def safe_zscore(x):
    x = np.asarray(x, dtype=np.float64)
    mu = float(np.mean(x))
    sigma = float(np.std(x, ddof=0))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1e-12
    return (x - mu) / sigma, mu, sigma


def find_contiguous_segments(mask):
    mask = np.asarray(mask, dtype=bool)
    idx = np.flatnonzero(mask)
    if idx.size == 0:
        return []
    splits = np.where(np.diff(idx) > 1)[0] + 1
    groups = np.split(idx, splits)
    return [(int(group[0]), int(group[-1])) for group in groups]


def expand_segment(power, start, stop, lower_threshold):
    start = int(start)
    stop = int(stop)
    while start > 0 and power[start - 1] > lower_threshold:
        start -= 1
    while stop < (len(power) - 1) and power[stop + 1] > lower_threshold:
        stop += 1
    return start, stop


def merge_close_segments(segments, gap_samps):
    if not segments:
        return []
    segments = sorted((int(start), int(stop)) for start, stop in segments)
    merged = [segments[0]]
    for start, stop in segments[1:]:
        prev_start, prev_stop = merged[-1]
        gap = start - prev_stop - 1
        if gap <= gap_samps:
            merged[-1] = (prev_start, max(prev_stop, stop))
        else:
            merged.append((start, stop))
    return merged


def duration_rejection_reason(length_samps, sfreq, min_duration_s=RIPPLE_MIN_DURATION_S, max_duration_s=RIPPLE_MAX_DURATION_S):
    duration_s = float(length_samps) / float(sfreq)
    if duration_s < min_duration_s:
        return "too_short"
    if duration_s > max_duration_s:
        return "too_long"
    return None


def nearest_trough_sample(trace, center_sample, start, stop):
    segment = np.asarray(trace[start: stop + 1], dtype=np.float64)
    if segment.size == 0:
        return None
    troughs, _ = find_peaks(-segment)
    if troughs.size == 0:
        return int(start + np.argmin(segment))
    troughs = troughs + int(start)
    return int(troughs[np.argmin(np.abs(troughs - int(center_sample)))])


def _load_behavior_subject_sets(load_dir=LOAD_DIR, condition=CONDITION):
    learn_df = pd.read_csv(load_dir / "learn_df_RWH.csv")
    combined_df = pd.read_csv(load_dir / "full_df_RWH.csv")
    feedback_subject_ids = sorted(learn_df["participant"].dropna().astype(str).unique().tolist())
    cue_subject_ids = sorted(
        combined_df.loc[combined_df["condition"] == condition, "participant"].dropna().astype(str).unique().tolist()
    )
    return {
        "feedback_start": feedback_subject_ids,
        "cue_on": cue_subject_ids,
        "union": sorted(set(feedback_subject_ids) | set(cue_subject_ids)),
    }


def discover_usable_subject_ids(ephys_dir=EPHYS_ANALYSES_DIR, condition=CONDITION):
    behavior_subject_sets = _load_behavior_subject_sets(load_dir=LOAD_DIR, condition=condition)
    available_subject_ids = {subject_path.name for subject_path in ephys_dir.iterdir() if subject_path.is_dir()}
    usable_subject_ids = []
    qc_rows = []
    for subject_id in behavior_subject_sets["union"]:
        if subject_id not in available_subject_ids:
            qc_rows.append({
                "participant": subject_id,
                "event": np.nan,
                "status": "skipped",
                "reason": "missing_ephys_subject_dir",
                "n_hpc_electrodes": np.nan,
                "n_valid_trials": np.nan,
            })
            continue
        subject_path = ephys_dir / subject_id
        neural_dir = subject_path / "neural" / condition
        elec_path = subject_path / f"{condition}_reref_elec_df"
        if not neural_dir.exists():
            qc_rows.append({
                "participant": subject_id,
                "event": np.nan,
                "status": "skipped",
                "reason": f"missing_{condition}_neural_dir",
                "n_hpc_electrodes": np.nan,
                "n_valid_trials": np.nan,
            })
            continue
        if not elec_path.exists():
            qc_rows.append({
                "participant": subject_id,
                "event": np.nan,
                "status": "skipped",
                "reason": f"missing_{condition}_reref_elec_df",
                "n_hpc_electrodes": np.nan,
                "n_valid_trials": np.nan,
            })
            continue
        elec_df = pd.read_csv(elec_path)
        if "label" not in elec_df.columns or "salman_region" not in elec_df.columns:
            qc_rows.append({
                "participant": subject_id,
                "event": np.nan,
                "status": "skipped",
                "reason": "invalid_electrode_metadata",
                "n_hpc_electrodes": np.nan,
                "n_valid_trials": np.nan,
            })
            continue
        labels = elec_df["label"].astype(str)
        hpc_df = elec_df[(elec_df["salman_region"] == "HPC") & (~labels.str.contains("lesion", case=False, na=False))].copy()
        if hpc_df.empty:
            qc_rows.append({
                "participant": subject_id,
                "event": np.nan,
                "status": "skipped",
                "reason": "no_nonlesion_hpc_electrodes",
                "n_hpc_electrodes": 0,
                "n_valid_trials": np.nan,
            })
            continue
        usable_subject_ids.append(subject_id)
        qc_rows.append({
            "participant": subject_id,
            "event": np.nan,
            "status": "processed",
            "reason": "discovered_from_behavior_tables",
            "n_hpc_electrodes": int(hpc_df.shape[0]),
            "n_valid_trials": np.nan,
        })
    return usable_subject_ids, pd.DataFrame.from_records(qc_rows, columns=QC_COLUMNS), behavior_subject_sets


def load_subject_event_data(subject_obj, event_name, event_cfg):
    epochs_path = (
        EPHYS_ANALYSES_DIR
        / subject_obj.subject
        / "neural"
        / CONDITION
        / f"{event_name}-epo.fif"
    )
    if not epochs_path.exists():
        raise FileNotFoundError(f"Missing epochs file: {epochs_path}")
    epochs = mne.read_epochs(epochs_path, preload=True, verbose=False)
    sfreq = float(epochs.info["sfreq"])
    if (sfreq / 2.0) <= RIPPLE_DETECTION_BAND[1]:
        raise ValueError(
            f"Sampling rate {sfreq} Hz cannot represent {RIPPLE_DETECTION_BAND[1]} Hz ripple band."
        )

    elec_df = subject_obj.elec_df.copy()
    elec_df["label"] = elec_df["label"].astype(str)
    hpc_df = elec_df[
        (elec_df["salman_region"] == "HPC")
        & (~elec_df["label"].str.contains("lesion", case=False, na=False))
    ].copy()
    if hpc_df.empty:
        raise ValueError(f"No hippocampal electrodes found for {subject_obj.subject}.")
    valid_labels = [label for label in hpc_df["label"].tolist() if label in epochs.ch_names]
    if not valid_labels:
        raise ValueError(f"No hippocampal electrodes present in {event_name} epochs for {subject_obj.subject}.")

    combined = subject_obj.combined_df.copy()
    combined = combined[
        (combined["participant"] == subject_obj.subject)
        & (combined["condition"] == CONDITION)
    ].copy()
    trial_col = event_cfg["trial_column"]
    trial_values = pd.to_numeric(combined[trial_col], errors="coerce")
    valid = trial_values.notna() & (trial_values >= 1) & (trial_values <= len(epochs))
    aligned_meta = combined.loc[valid].copy()
    aligned_meta[trial_col] = pd.to_numeric(aligned_meta[trial_col], errors="coerce").astype(int)
    aligned_meta = aligned_meta.sort_values(trial_col).reset_index(drop=True)
    trial_numbers = aligned_meta[trial_col].to_numpy(dtype=np.int64)
    trial_indices_zero_based = trial_numbers - 1
    if len(trial_numbers) < MIN_VALID_TRIALS:
        raise ValueError(
            f"{event_name} has only {len(trial_numbers)} aligned valid trials for {subject_obj.subject}."
        )

    aligned_epochs_full = epochs[trial_indices_zero_based].copy().pick(valid_labels)
    full_raw_data = aligned_epochs_full.get_data(copy=True)
    full_times = aligned_epochs_full.times.astype(np.float64)
    aligned_epochs_detection = aligned_epochs_full.copy().crop(
        tmin=float(event_cfg["window"][0]),
        tmax=float(event_cfg["window"][1]),
        include_tmax=True,
    )
    detection_data = aligned_epochs_detection.get_data(copy=True)
    detection_times = aligned_epochs_detection.times.astype(np.float64)

    hpc_lookup = hpc_df.set_index("label")
    electrode_region = np.asarray([hpc_lookup.loc[label, "salman_region"] for label in valid_labels], dtype=object)
    electrode_hemi = np.asarray([
        str(hpc_lookup.loc[label, "hemisphere"]).lower() if pd.notna(hpc_lookup.loc[label, "hemisphere"]) else np.nan
        for label in valid_labels
    ], dtype=object)

    return {
        "event": event_name,
        "trial_column": trial_col,
        "trial_window_s": float(event_cfg["window"][1] - event_cfg["window"][0]),
        "sfreq": sfreq,
        "n_hpc_electrodes": int(len(valid_labels)),
        "full_raw_data": np.asarray(full_raw_data, dtype=np.float64),
        "full_times": full_times,
        "detection_data": np.asarray(detection_data, dtype=np.float64),
        "detection_times": detection_times,
        "trial_numbers": trial_numbers,
        "trial_indices_zero_based": trial_indices_zero_based,
        "metadata": aligned_meta,
        "electrode_labels": np.asarray(valid_labels, dtype=object),
        "electrode_region": electrode_region,
        "electrode_hemi": electrode_hemi,
    }


def detect_epochwise_ieds(smoothed_power, times, sfreq, trial_numbers, trial_indices_zero_based, participant, event, electrode, electrode_region, electrode_hemi):
    flat_z, mean_value, std_value = safe_zscore(smoothed_power.reshape(-1))
    z_trials = flat_z.reshape(smoothed_power.shape)
    records = []
    for trial_pos, trial_number in enumerate(trial_numbers):
        trial_z = z_trials[trial_pos]
        segments = find_contiguous_segments(trial_z > IED_Z_THRESHOLD)
        for start, stop in segments:
            peak_sample = int(start + np.argmax(trial_z[start: stop + 1]))
            records.append(
                {
                    "participant": participant,
                    "event": event,
                    "trial_number": int(trial_number),
                    "trial_index_zero_based": int(trial_indices_zero_based[trial_pos]),
                    "electrode": str(electrode),
                    "electrode_region": str(electrode_region),
                    "electrode_hemi": electrode_hemi,
                    "ied_peak_s": float(times[peak_sample]),
                    "ied_peak_sample": int(peak_sample),
                    "ied_peak_z": float(trial_z[peak_sample]),
                }
            )
    return pd.DataFrame.from_records(records, columns=IED_COLUMNS), {
        "z_trials": z_trials,
        "mean_value": float(mean_value),
        "std_value": float(std_value),
    }


def detect_epochwise_ripples(detection_envelope, ripple_trace, times, sfreq, trial_numbers, trial_indices_zero_based, participant, event, trial_column, electrode, electrode_region, electrode_hemi, ied_events_df):
    robust_mean, robust_sd = lms_loc_scale(detection_envelope.reshape(-1))
    clip_value = robust_mean + (RIPPLE_CLIP_SD * robust_sd)
    clipped_envelope = np.minimum(detection_envelope, clip_value)
    squared_power = clipped_envelope ** 2
    smoothed_power, _ = smooth_power_kaiser(squared_power, sfreq=sfreq, cutoff_hz=SMOOTH_CUTOFF_HZ)
    flat_smoothed = smoothed_power.reshape(-1)
    smoothed_mean = float(np.mean(flat_smoothed))
    smoothed_sd = float(np.std(flat_smoothed, ddof=0))
    if not np.isfinite(smoothed_sd) or smoothed_sd <= 0:
        smoothed_sd = 1e-12
    upper_threshold = smoothed_mean + (RIPPLE_UPPER_SD * smoothed_sd)
    lower_threshold = smoothed_mean + (RIPPLE_LOWER_SD * smoothed_sd)
    gap_samps = int(round(RIPPLE_MERGE_GAP_S * sfreq))

    candidate_rows = []
    retained_rows = []

    for trial_pos, trial_number in enumerate(trial_numbers):
        trial_power = smoothed_power[trial_pos]
        trial_trace = ripple_trace[trial_pos]
        raw_segments = find_contiguous_segments(trial_power > upper_threshold)
        expanded = [expand_segment(trial_power, start, stop, lower_threshold) for start, stop in raw_segments]
        merged = merge_close_segments(expanded, gap_samps=gap_samps)

        trial_ieds = ied_events_df[
            (ied_events_df["participant"] == participant)
            & (ied_events_df["event"] == event)
            & (ied_events_df["electrode"] == electrode)
            & (ied_events_df["trial_number"] == int(trial_number))
        ]
        ied_times = trial_ieds["ied_peak_s"].to_numpy(dtype=np.float64) if not trial_ieds.empty else np.array([], dtype=np.float64)

        for start, stop in merged:
            peak_power_sample = int(start + np.argmax(trial_power[start: stop + 1]))
            peak_trace_sample = nearest_trough_sample(trial_trace, peak_power_sample, start, stop)
            duration_samps = int(stop - start + 1)
            rejection_reason = duration_rejection_reason(duration_samps, sfreq=sfreq)
            if peak_trace_sample is None:
                rejection_reason = "no_trough_found"

            nearest_ied_delta_s = np.nan
            ied_within_100ms = False
            if peak_trace_sample is not None and ied_times.size > 0:
                nearest_ied_delta_s = float(np.min(np.abs(ied_times - float(times[peak_trace_sample]))))
                ied_within_100ms = bool(nearest_ied_delta_s <= IED_EXCLUSION_WINDOW_S)
                if rejection_reason is None and ied_within_100ms:
                    rejection_reason = "ied_within_100ms"

            row = {
                "participant": participant,
                "event": event,
                "condition": CONDITION,
                "trial_column": trial_column,
                "trial_number": int(trial_number),
                "trial_index_zero_based": int(trial_indices_zero_based[trial_pos]),
                "electrode": str(electrode),
                "electrode_region": str(electrode_region),
                "electrode_hemi": electrode_hemi,
                "ripple_start_s": float(times[start]),
                "ripple_end_s": float(times[stop]),
                "ripple_peak_s": float(times[peak_trace_sample]) if peak_trace_sample is not None else np.nan,
                "ripple_duration_s": float(duration_samps / sfreq),
                "ripple_start_sample": int(start),
                "ripple_end_sample": int(stop),
                "ripple_peak_sample": int(peak_trace_sample) if peak_trace_sample is not None else np.nan,
                "peak_smoothed_power": float(trial_power[peak_power_sample]),
                "nearest_ied_delta_s": nearest_ied_delta_s,
                "ied_within_100ms": bool(ied_within_100ms),
                "retained_after_ied_exclusion": rejection_reason is None,
                "rejection_reason": rejection_reason,
            }
            candidate_rows.append(row)
            if rejection_reason is None:
                retained_rows.append({key: row[key] for key in RIPPLE_EVENT_COLUMNS})

    debug = {
        "robust_mean": float(robust_mean),
        "robust_sd": float(robust_sd),
        "clip_value": float(clip_value),
        "smoothed_mean": float(smoothed_mean),
        "smoothed_sd": float(smoothed_sd),
        "upper_threshold": float(upper_threshold),
        "lower_threshold": float(lower_threshold),
        "smoothed_power": smoothed_power,
    }
    return (
        pd.DataFrame.from_records(candidate_rows, columns=RIPPLE_CANDIDATE_COLUMNS),
        pd.DataFrame.from_records(retained_rows, columns=RIPPLE_EVENT_COLUMNS),
        debug,
    )


def _build_trial_ripple_rates(participant, event_name, state, ied_events_df, ripple_events_df):
    trial_numbers = state["trial_numbers"]
    trial_indices_zero_based = state["trial_indices_zero_based"]
    trial_column = state["trial_column"]
    trial_window_s = state["trial_window_s"]
    n_hpc_electrodes = state["n_hpc_electrodes"]

    if ied_events_df.empty:
        ied_counts = pd.Series(dtype=np.int64)
    else:
        ied_counts = ied_events_df.groupby("trial_number").size()
    if ripple_events_df.empty:
        ripple_counts = pd.Series(dtype=np.int64)
    else:
        ripple_counts = ripple_events_df.groupby("trial_number").size()

    rows = []
    for trial_number, trial_idx in zip(trial_numbers, trial_indices_zero_based):
        n_ripples = int(ripple_counts.get(int(trial_number), 0))
        n_ieds = int(ied_counts.get(int(trial_number), 0))
        rows.append({
            "participant": participant,
            "event": event_name,
            "condition": CONDITION,
            "trial_number": int(trial_number),
            "trial_index_zero_based": int(trial_idx),
            "trial_column": trial_column,
            "n_ripples": n_ripples,
            "n_ieds": n_ieds,
            "n_hpc_electrodes": int(n_hpc_electrodes),
            "trial_window_s": float(trial_window_s),
            "ripple_rate_hz": float(n_ripples / trial_window_s),
        })
    return pd.DataFrame.from_records(rows, columns=TRIAL_RATE_COLUMNS)


def _build_electrode_ripple_rates(participant, event_name, state, ied_events_df, ripple_events_df):
    rows = []
    n_trials = int(len(state["trial_numbers"]))
    trial_window_s = float(state["trial_window_s"])
    labels = state["electrode_labels"]
    regions = state["electrode_region"]
    hemis = state["electrode_hemi"]

    ripple_counts = ripple_events_df.groupby("electrode").size() if not ripple_events_df.empty else pd.Series(dtype=np.int64)
    ied_counts = ied_events_df.groupby("electrode").size() if not ied_events_df.empty else pd.Series(dtype=np.int64)

    for label, region, hemi in zip(labels, regions, hemis):
        n_ripples = int(ripple_counts.get(str(label), 0))
        n_ieds = int(ied_counts.get(str(label), 0))
        rows.append({
            "participant": participant,
            "event": event_name,
            "condition": CONDITION,
            "electrode": str(label),
            "electrode_region": str(region),
            "electrode_hemi": hemi,
            "n_trials": n_trials,
            "n_ripples": n_ripples,
            "n_ieds": n_ieds,
            "mean_ripples_per_trial": float(n_ripples / n_trials),
            "mean_ripple_rate_hz": float((n_ripples / n_trials) / trial_window_s),
        })
    return pd.DataFrame.from_records(rows, columns=ELECTRODE_RATE_COLUMNS)


def _subject_result_stub(subject_id, qc_rows=None):
    return {
        "subject_id": subject_id,
        "ied_events_df": _empty_df(IED_COLUMNS),
        "ripple_candidates_df": _empty_df(RIPPLE_CANDIDATE_COLUMNS),
        "ripple_events_df": _empty_df(RIPPLE_EVENT_COLUMNS),
        "trial_ripple_rates_df": _empty_df(TRIAL_RATE_COLUMNS),
        "electrode_ripple_rates_df": _empty_df(ELECTRODE_RATE_COLUMNS),
        "qc_df": pd.DataFrame.from_records(qc_rows or [], columns=QC_COLUMNS),
        "summary_figures": {},
        "event_payloads": {},
        "debug": {"events": {}},
    }


def make_ripple_example_panels(results, n_examples=TOP_N_EXAMPLES, context_s=EXAMPLE_CONTEXT_S):
    ripple_events_df = results["ripple_events_df"]
    if ripple_events_df.empty:
        print("No retained ripples available for example panels.")
        return None
    if not results.get("event_payloads"):
        raise ValueError("Example panels require event_payloads. Re-run with make_plots=True.")

    example_df = ripple_events_df.sort_values("peak_smoothed_power", ascending=False).head(int(n_examples)).reset_index(drop=True)
    n_examples = len(example_df)
    fig, axes = plt.subplots(n_examples, 2, figsize=(14, max(4, 3 * n_examples)), sharex=False)
    if n_examples == 1:
        axes = np.asarray([axes])

    for row_idx, row in example_df.iterrows():
        event_payload = results["event_payloads"][row["event"]]
        trial_pos = event_payload["trial_lookup"][int(row["trial_number"])]
        ch_pos = event_payload["electrode_lookup"][row["electrode"]]
        full_times = event_payload["full_times"]
        raw_trace = event_payload["full_raw_data"][trial_pos, ch_pos]
        ripple_trace = event_payload["full_ripple_trace"][trial_pos, ch_pos]
        center_sample = int(np.argmin(np.abs(full_times - float(row["ripple_peak_s"]))))
        half_window = int(round(context_s * event_payload["sfreq"]))
        start = max(0, center_sample - half_window)
        stop = min(len(full_times), center_sample + half_window + 1)
        plot_times = full_times[start:stop]
        raw_trace_uv = raw_trace[start:stop] * UV_SCALE
        ripple_trace_uv = ripple_trace[start:stop] * UV_SCALE

        ax_raw = axes[row_idx, 0]
        ax_ripple = axes[row_idx, 1]
        ax_raw.plot(plot_times, raw_trace_uv, color="black", linewidth=1.2)
        ax_ripple.plot(plot_times, ripple_trace_uv, color="dimgray", linewidth=1.2)

        for ax in (ax_raw, ax_ripple):
            ax.axvspan(float(row["ripple_start_s"]), float(row["ripple_end_s"]), color="0.85", alpha=0.8)
            ax.axvline(float(row["ripple_peak_s"]), color="tab:red", linestyle="--", linewidth=1.0)
            ax.set_xlim(plot_times[0], plot_times[-1])

        ax_raw.set_title(f"Raw LFP | {row['event']} | {row['electrode']} | trial {int(row['trial_number'])}")
        ax_ripple.set_title(f"70-180 Hz trace | peak power {row['peak_smoothed_power']:.2f}")
        ax_raw.set_ylabel("Amplitude (uV)")
        ax_ripple.set_ylabel("Amplitude (uV)")
        ax_raw.set_xlabel("Time from event onset (s)")
        ax_ripple.set_xlabel("Time from event onset (s)")

    fig.tight_layout()
    plt.show()
    return fig


def _collect_locked_snippets(results, plot_window_s=SUMMARY_WINDOW_S, compute_window_s=SUMMARY_COMPUTE_WINDOW_S, groupby_event=True):
    ripple_events_df = results["ripple_events_df"]
    if ripple_events_df.empty:
        return None
    if not results.get("event_payloads"):
        raise ValueError("Ripple-locked summaries require event_payloads. Re-run with make_plots=True.")
    if compute_window_s <= plot_window_s:
        raise ValueError("compute_window_s must be larger than plot_window_s for padded TFRs.")

    grouped_payloads = {}
    grouped_events = [
        (event_name, ripple_events_df[ripple_events_df["event"] == event_name].copy())
        for event_name in EVENT_SPECS
    ] if groupby_event else [("pooled", ripple_events_df.copy())]

    for group_name, group_df in grouped_events:
        if group_df.empty:
            continue

        raw_compute_snippets = []
        ripple_compute_snippets = []
        metadata_rows = []
        group_sfreq = None
        compute_time_axis = None

        for _, row in group_df.iterrows():
            event_payload = results["event_payloads"][row["event"]]
            sfreq = float(event_payload["sfreq"])
            if group_sfreq is None:
                group_sfreq = sfreq
            elif not np.isclose(group_sfreq, sfreq):
                raise ValueError(f"Mixed sampling rates found within summary group {group_name}.")

            half_window = int(round(compute_window_s * sfreq))
            full_times = event_payload["full_times"]
            center_sample = int(np.argmin(np.abs(full_times - float(row["ripple_peak_s"]))))
            if center_sample - half_window < 0 or center_sample + half_window >= len(full_times):
                continue

            trial_pos = event_payload["trial_lookup"][int(row["trial_number"])]
            ch_pos = event_payload["electrode_lookup"][row["electrode"]]
            raw_compute = event_payload["full_raw_data"][trial_pos, ch_pos, center_sample - half_window: center_sample + half_window + 1]
            ripple_compute = event_payload["full_ripple_trace"][trial_pos, ch_pos, center_sample - half_window: center_sample + half_window + 1]

            if compute_time_axis is None:
                compute_time_axis = np.arange(-half_window, half_window + 1, dtype=np.float64) / sfreq

            raw_compute_snippets.append(raw_compute)
            ripple_compute_snippets.append(ripple_compute)
            metadata_rows.append({
                "participant": row["participant"],
                "event": row["event"],
                "trial_number": int(row["trial_number"]),
                "electrode": row["electrode"],
                "peak_smoothed_power": float(row["peak_smoothed_power"]),
            })

        if not raw_compute_snippets:
            continue

        plot_mask = (compute_time_axis >= -plot_window_s) & (compute_time_axis <= plot_window_s)
        plot_time_axis = compute_time_axis[plot_mask]
        if plot_time_axis.size == 0:
            raise ValueError(f"No plotting samples were retained for {group_name}.")
        if not (np.isclose(plot_time_axis[0], -plot_window_s) and np.isclose(plot_time_axis[-1], plot_window_s)):
            raise ValueError(f"Plot time axis does not match requested window for {group_name}.")

        grouped_payloads[group_name] = {
            "raw_compute": np.asarray(raw_compute_snippets, dtype=np.float64),
            "ripple_compute": np.asarray(ripple_compute_snippets, dtype=np.float64),
            "compute_time_axis": compute_time_axis,
            "plot_mask": plot_mask,
            "plot_time_axis": plot_time_axis,
            "sfreq": float(group_sfreq),
            "metadata": pd.DataFrame.from_records(metadata_rows),
        }

    return grouped_payloads or None


def _compute_baseline_z_tfr(raw_compute, sfreq, compute_time_axis, plot_mask, plot_window_s=SUMMARY_WINDOW_S, baseline_window=TFR_BASELINE_WINDOW):
    fmax = min(250, math.floor(0.45 * sfreq))
    freqs = np.arange(20, fmax + 1, 5, dtype=np.float64)
    n_cycles = np.clip(freqs / 20.0, 3.0, 10.0)
    demeaned = raw_compute - np.mean(raw_compute, axis=-1, keepdims=True)

    power = mne.time_frequency.tfr_array_morlet(
        demeaned[:, np.newaxis, :],
        sfreq=sfreq,
        freqs=freqs,
        n_cycles=n_cycles,
        output="power",
        zero_mean=True,
        use_fft=True,
        decim=1,
        n_jobs=1,
        verbose=False,
    )[:, 0, :, :]

    baseline_mask = (compute_time_axis >= baseline_window[0]) & (compute_time_axis <= baseline_window[1])
    if not np.any(baseline_mask):
        raise ValueError("Baseline window does not overlap the padded ripple snippets.")

    baseline_power = power[:, :, baseline_mask]
    baseline_mean = np.mean(baseline_power, axis=-1, keepdims=True)
    baseline_sd = np.std(baseline_power, axis=-1, keepdims=True)
    baseline_sd[~np.isfinite(baseline_sd) | (baseline_sd <= 0)] = 1e-12
    z_power = (power - baseline_mean) / baseline_sd

    plot_time_axis = compute_time_axis[plot_mask]
    if plot_time_axis.size == 0:
        raise ValueError("Plot mask removed all spectrogram samples.")
    if not (np.isclose(plot_time_axis[0], -plot_window_s) and np.isclose(plot_time_axis[-1], plot_window_s)):
        raise ValueError("Spectrogram plot time axis does not match the requested display window.")

    return {
        "z_power": z_power[:, :, plot_mask],
        "freqs": freqs,
        "plot_time_axis": plot_time_axis,
    }


def _summarize_tfr(z_power, summary_stat="median"):
    if summary_stat == "median":
        return np.median(z_power, axis=0)
    if summary_stat == "mean":
        return np.mean(z_power, axis=0)
    raise ValueError(f"Unsupported summary_stat: {summary_stat}")


def _compute_event_color_limits(pooled_tfr, electrode_tfrs, lower_pct=2.0, upper_pct=98.0):
    arrays = [np.asarray(pooled_tfr, dtype=np.float64)]
    arrays.extend(np.asarray(arr, dtype=np.float64) for arr in electrode_tfrs if arr is not None)
    flat = np.concatenate([arr.ravel() for arr in arrays if arr.size > 0])
    flat = flat[np.isfinite(flat)]
    if flat.size == 0:
        return -1.0, 1.0
    vmin, vmax = np.percentile(flat, [lower_pct, upper_pct])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmax = float(np.max(np.abs(flat))) if flat.size else 1.0
        if not np.isfinite(vmax) or vmax <= 0:
            vmax = 1.0
        vmin = -vmax
    return float(vmin), float(vmax)


def _compute_peak_frequency_diagnostics(z_power, freqs, time_axis, center_half_width_s=0.01):
    center_mask = np.abs(time_axis) <= float(center_half_width_s)
    if not np.any(center_mask):
        center_idx = int(np.argmin(np.abs(time_axis)))
        center_mask = np.zeros_like(time_axis, dtype=bool)
        center_mask[max(0, center_idx - 1): min(len(time_axis), center_idx + 2)] = True

    center_power = np.mean(z_power[:, :, center_mask], axis=-1)
    peak_idx = np.argmax(center_power, axis=1)
    peak_freqs = freqs[peak_idx]
    quantiles = np.quantile(peak_freqs, [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0])
    band_counts = {
        "20_40": int(np.sum((peak_freqs >= 20) & (peak_freqs < 40))),
        "40_70": int(np.sum((peak_freqs >= 40) & (peak_freqs < 70))),
        "70_120": int(np.sum((peak_freqs >= 70) & (peak_freqs < 120))),
        "120_180": int(np.sum((peak_freqs >= 120) & (peak_freqs < 180))),
        "180_225": int(np.sum((peak_freqs >= 180) & (peak_freqs <= 225))),
    }
    return {
        "peak_freqs_hz": peak_freqs,
        "peak_freq_quantiles_hz": quantiles,
        "peak_freq_band_counts": band_counts,
    }


def make_ripple_locked_summary(results, plot_window_s=SUMMARY_WINDOW_S, compute_window_s=SUMMARY_COMPUTE_WINDOW_S, baseline_window=TFR_BASELINE_WINDOW, groupby_event=True, summary_stat="median"):
    grouped_snippets = _collect_locked_snippets(
        results,
        plot_window_s=plot_window_s,
        compute_window_s=compute_window_s,
        groupby_event=groupby_event,
    )
    if grouped_snippets is None:
        print("No retained ripples with full plotting context were available.")
        return {}

    summary_results = {}
    for group_name, snippet_payload in grouped_snippets.items():
        raw_compute = snippet_payload["raw_compute"]
        ripple_compute = snippet_payload["ripple_compute"]
        compute_time_axis = snippet_payload["compute_time_axis"]
        plot_mask = snippet_payload["plot_mask"]
        plot_time_axis = snippet_payload["plot_time_axis"]
        sfreq = snippet_payload["sfreq"]
        meta = snippet_payload["metadata"]
        group_label = group_name.replace("_", " ")

        raw_plot_uv = raw_compute[:, plot_mask] * UV_SCALE
        ripple_plot_uv = ripple_compute[:, plot_mask] * UV_SCALE

        fig_avg, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
        axes[0].plot(plot_time_axis, np.mean(raw_plot_uv, axis=0), color="black", linewidth=2)
        axes[0].axvline(0.0, color="tab:red", linestyle="--", linewidth=1.0)
        axes[0].set_title(f"{group_label} ripple-locked raw LFP (n={raw_plot_uv.shape[0]})")
        axes[0].set_xlabel("Time from ripple peak (s)")
        axes[0].set_ylabel("Amplitude (uV)")
        axes[1].plot(plot_time_axis, np.mean(ripple_plot_uv, axis=0), color="dimgray", linewidth=2)
        axes[1].axvline(0.0, color="tab:red", linestyle="--", linewidth=1.0)
        axes[1].set_title(f"{group_label} ripple-locked 70-180 Hz trace")
        axes[1].set_xlabel("Time from ripple peak (s)")
        axes[1].set_ylabel("Amplitude (uV)")
        fig_avg.tight_layout()
        plt.show()

        tfr_payload = _compute_baseline_z_tfr(
            raw_compute,
            sfreq=sfreq,
            compute_time_axis=compute_time_axis,
            plot_mask=plot_mask,
            plot_window_s=plot_window_s,
            baseline_window=baseline_window,
        )
        pooled_summary = _summarize_tfr(tfr_payload["z_power"], summary_stat=summary_stat)

        per_electrode_counts = meta.groupby("electrode").size()
        eligible = per_electrode_counts[per_electrode_counts >= MIN_ELECTRODE_SUMMARY_RIPPLES].index.tolist()
        electrode_summaries = {}
        for electrode in eligible:
            keep = meta.index[meta["electrode"] == electrode].to_numpy(dtype=np.int64)
            electrode_summaries[electrode] = _summarize_tfr(tfr_payload["z_power"][keep], summary_stat=summary_stat)

        vmin, vmax = _compute_event_color_limits(pooled_summary, list(electrode_summaries.values()))

        fig_spec, ax = plt.subplots(figsize=(8, 6))
        mesh = ax.pcolormesh(
            tfr_payload["plot_time_axis"],
            tfr_payload["freqs"],
            pooled_summary,
            shading="auto",
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
        )
        ax.axvline(0.0, color="white", linestyle="--", linewidth=1.0)
        ax.set_title(f"{results['subject_id']} {group_label} ripple-locked spectrogram (n={raw_compute.shape[0]})")
        ax.set_xlabel("Time from ripple peak (s)")
        ax.set_ylabel("Frequency (Hz)")
        cbar = fig_spec.colorbar(mesh, ax=ax)
        cbar.set_label("z")
        fig_spec.tight_layout()
        plt.show()

        electrode_fig = None
        if eligible:
            n_cols = min(2, len(eligible))
            n_rows = int(math.ceil(len(eligible) / n_cols))
            electrode_fig, electrode_axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4.5 * n_rows), squeeze=False)
            axes_flat = electrode_axes.ravel()
            for ax, electrode in zip(axes_flat, eligible):
                ax.pcolormesh(
                    tfr_payload["plot_time_axis"],
                    tfr_payload["freqs"],
                    electrode_summaries[electrode],
                    shading="auto",
                    cmap="viridis",
                    vmin=vmin,
                    vmax=vmax,
                )
                ax.axvline(0.0, color="white", linestyle="--", linewidth=1.0)
                ax.set_title(f"{electrode} (n={int(per_electrode_counts[electrode])})")
                ax.set_xlabel("Time from ripple peak (s)")
                ax.set_ylabel("Frequency (Hz)")
            for ax in axes_flat[len(eligible):]:
                ax.axis("off")
            electrode_fig.tight_layout()
            plt.show()

        diagnostics = _compute_peak_frequency_diagnostics(
            tfr_payload["z_power"],
            freqs=tfr_payload["freqs"],
            time_axis=tfr_payload["plot_time_axis"],
        )
        print(f"{group_name} ripple peak frequency quantiles (Hz): {np.round(diagnostics['peak_freq_quantiles_hz'], 2).tolist()}")
        print(f"{group_name} ripple peak frequency band counts: {diagnostics['peak_freq_band_counts']}")

        summary_results[group_name] = {
            "snippet_payload": snippet_payload,
            "tfr_payload": tfr_payload,
            "spectrogram_summary_stat": summary_stat,
            "color_limits": (vmin, vmax),
            "peak_frequency_diagnostics": diagnostics,
            "grand_average_fig": fig_avg,
            "pooled_spectrogram_fig": fig_spec,
            "electrode_spectrogram_fig": electrode_fig,
        }

    return summary_results


def run_subject_ripple_pipeline(subject_id=SUBJECT_ID, make_plots=False):
    subject_qc_rows = []
    try:
        subject_obj = Subject(subject_id, str(BASE_DIR), str(LOAD_DIR), min_trials=MIN_VALID_TRIALS, verbose=False)
    except Exception as exc:
        subject_qc_rows.append({
            "participant": subject_id,
            "event": np.nan,
            "status": "skipped",
            "reason": f"subject_init_failed: {exc}",
            "n_hpc_electrodes": np.nan,
            "n_valid_trials": np.nan,
        })
        return _subject_result_stub(subject_id, qc_rows=subject_qc_rows)

    event_payloads = {}
    ied_frames = []
    ripple_candidate_frames = []
    ripple_event_frames = []
    trial_rate_frames = []
    electrode_rate_frames = []
    debug = {"events": {}}
    processed_events = 0

    for event_name, event_cfg in EVENT_SPECS.items():
        try:
            state = load_subject_event_data(subject_obj, event_name, event_cfg)
        except Exception as exc:
            subject_qc_rows.append({
                "participant": subject_id,
                "event": event_name,
                "status": "skipped",
                "reason": str(exc),
                "n_hpc_electrodes": np.nan,
                "n_valid_trials": np.nan,
            })
            continue

        sfreq = state["sfreq"]
        detection_data = state["detection_data"]
        full_raw_data = state["full_raw_data"]
        detection_times = state["detection_times"]
        trial_numbers = state["trial_numbers"]
        trial_indices_zero_based = state["trial_indices_zero_based"]
        labels = state["electrode_labels"]
        regions = state["electrode_region"]
        hemis = state["electrode_hemi"]

        ied_filtered, _, ied_envelope = compute_fast_envelope(detection_data, sfreq=sfreq, band=IED_BAND, fir_window="hamming")
        ied_power = ied_envelope ** 2
        ied_smoothed, _ = smooth_power_kaiser(ied_power, sfreq=sfreq, cutoff_hz=SMOOTH_CUTOFF_HZ)

        _, _, ripple_detection_envelope = compute_fast_envelope(detection_data, sfreq=sfreq, band=RIPPLE_DETECTION_BAND)
        full_ripple_trace = bandpass_filter(full_raw_data, sfreq=sfreq, band=RIPPLE_TRACE_BAND)
        detection_ripple_trace = bandpass_filter(detection_data, sfreq=sfreq, band=RIPPLE_TRACE_BAND)

        event_ied_frames = []
        event_debug = {
            "sfreq": sfreq,
            "trial_count": int(len(trial_numbers)),
            "ied": {},
            "ripple": {},
        }

        for ch_idx, label in enumerate(labels):
            ied_df_ch, ied_debug = detect_epochwise_ieds(
                smoothed_power=ied_smoothed[:, ch_idx, :],
                times=detection_times,
                sfreq=sfreq,
                trial_numbers=trial_numbers,
                trial_indices_zero_based=trial_indices_zero_based,
                participant=subject_id,
                event=event_name,
                electrode=str(label),
                electrode_region=str(regions[ch_idx]),
                electrode_hemi=hemis[ch_idx],
            )
            event_ied_frames.append(ied_df_ch)
            event_debug["ied"][str(label)] = ied_debug

        event_ied_df = pd.concat([df for df in event_ied_frames if not df.empty], ignore_index=True) if event_ied_frames else _empty_df(IED_COLUMNS)
        ied_frames.append(event_ied_df)

        event_candidate_frames = []
        event_ripple_frames = []
        for ch_idx, label in enumerate(labels):
            candidate_df_ch, ripple_df_ch, ripple_debug = detect_epochwise_ripples(
                detection_envelope=ripple_detection_envelope[:, ch_idx, :],
                ripple_trace=detection_ripple_trace[:, ch_idx, :],
                times=detection_times,
                sfreq=sfreq,
                trial_numbers=trial_numbers,
                trial_indices_zero_based=trial_indices_zero_based,
                participant=subject_id,
                event=event_name,
                trial_column=event_cfg["trial_column"],
                electrode=str(label),
                electrode_region=str(regions[ch_idx]),
                electrode_hemi=hemis[ch_idx],
                ied_events_df=event_ied_df,
            )
            event_candidate_frames.append(candidate_df_ch)
            event_ripple_frames.append(ripple_df_ch)
            ripple_candidate_frames.append(candidate_df_ch)
            ripple_event_frames.append(ripple_df_ch)
            event_debug["ripple"][str(label)] = ripple_debug

        event_ripple_df = pd.concat([df for df in event_ripple_frames if not df.empty], ignore_index=True) if event_ripple_frames else _empty_df(RIPPLE_EVENT_COLUMNS)
        trial_rate_frames.append(_build_trial_ripple_rates(subject_id, event_name, state, event_ied_df, event_ripple_df))
        electrode_rate_frames.append(_build_electrode_ripple_rates(subject_id, event_name, state, event_ied_df, event_ripple_df))

        if make_plots:
            event_payloads[event_name] = {
                "sfreq": sfreq,
                "full_raw_data": full_raw_data,
                "full_times": state["full_times"],
                "full_ripple_trace": full_ripple_trace,
                "trial_numbers": trial_numbers,
                "trial_lookup": {int(trial): idx for idx, trial in enumerate(trial_numbers.tolist())},
                "electrode_labels": labels,
                "electrode_lookup": {str(label): idx for idx, label in enumerate(labels.tolist())},
                "electrode_region": regions,
                "electrode_hemi": hemis,
                "metadata": state["metadata"],
            }

        debug["events"][event_name] = event_debug
        processed_events += 1
        subject_qc_rows.append({
            "participant": subject_id,
            "event": event_name,
            "status": "processed",
            "reason": "processed",
            "n_hpc_electrodes": int(state["n_hpc_electrodes"]),
            "n_valid_trials": int(len(trial_numbers)),
        })

    if processed_events == 0:
        return _subject_result_stub(subject_id, qc_rows=subject_qc_rows)

    if processed_events < len(EVENT_SPECS):
        subject_qc_rows.append({
            "participant": subject_id,
            "event": np.nan,
            "status": "partial",
            "reason": "one_or_more_events_skipped",
            "n_hpc_electrodes": np.nan,
            "n_valid_trials": np.nan,
        })

    ied_events_df = pd.concat([df for df in ied_frames if not df.empty], ignore_index=True) if ied_frames else _empty_df(IED_COLUMNS)
    ripple_candidates_df = pd.concat([df for df in ripple_candidate_frames if not df.empty], ignore_index=True) if ripple_candidate_frames else _empty_df(RIPPLE_CANDIDATE_COLUMNS)
    ripple_events_df = pd.concat([df for df in ripple_event_frames if not df.empty], ignore_index=True) if ripple_event_frames else _empty_df(RIPPLE_EVENT_COLUMNS)
    trial_ripple_rates_df = pd.concat(trial_rate_frames, ignore_index=True) if trial_rate_frames else _empty_df(TRIAL_RATE_COLUMNS)
    electrode_ripple_rates_df = pd.concat(electrode_rate_frames, ignore_index=True) if electrode_rate_frames else _empty_df(ELECTRODE_RATE_COLUMNS)
    qc_df = pd.DataFrame.from_records(subject_qc_rows, columns=QC_COLUMNS)

    results = {
        "subject_id": subject_id,
        "ied_events_df": ied_events_df,
        "ripple_candidates_df": ripple_candidates_df,
        "ripple_events_df": ripple_events_df,
        "trial_ripple_rates_df": trial_ripple_rates_df,
        "electrode_ripple_rates_df": electrode_ripple_rates_df,
        "qc_df": qc_df,
        "summary_figures": {},
        "event_payloads": event_payloads,
        "debug": debug,
    }

    if make_plots:
        results["summary_figures"] = make_ripple_locked_summary(
            results,
            plot_window_s=SUMMARY_WINDOW_S,
            compute_window_s=SUMMARY_COMPUTE_WINDOW_S,
            baseline_window=TFR_BASELINE_WINDOW,
            groupby_event=True,
            summary_stat="median",
        )

    return results


def run_ripple_pipeline_batch(subject_ids=None, n_jobs=-1, backend="loky", make_plots=False):
    discovered_subjects, discovery_qc_df, behavior_subject_sets = discover_usable_subject_ids()
    if subject_ids is None:
        subject_ids = discovered_subjects
    else:
        subject_ids = [subject_id for subject_id in subject_ids]

    backend_used = backend
    try:
        subject_results_list = Parallel(n_jobs=n_jobs, backend=backend_used)(
            delayed(run_subject_ripple_pipeline)(subject_id=subject_id, make_plots=make_plots)
            for subject_id in subject_ids
        )
    except PermissionError as exc:
        if backend_used != "loky":
            raise
        warnings.warn(f"loky backend unavailable in this environment ({exc}); retrying with threading backend.")
        backend_used = "threading"
        subject_results_list = Parallel(n_jobs=n_jobs, backend=backend_used)(
            delayed(run_subject_ripple_pipeline)(subject_id=subject_id, make_plots=make_plots)
            for subject_id in subject_ids
        )

    subject_results = {result["subject_id"]: result for result in subject_results_list}
    ripple_events_df = pd.concat(
        [result["ripple_events_df"] for result in subject_results_list if not result["ripple_events_df"].empty],
        ignore_index=True,
    ) if subject_results_list else _empty_df(RIPPLE_EVENT_COLUMNS)
    if ripple_events_df.empty:
        ripple_events_df = _empty_df(RIPPLE_EVENT_COLUMNS)

    trial_ripple_rates_df = pd.concat(
        [result["trial_ripple_rates_df"] for result in subject_results_list if not result["trial_ripple_rates_df"].empty],
        ignore_index=True,
    ) if subject_results_list else _empty_df(TRIAL_RATE_COLUMNS)
    if trial_ripple_rates_df.empty:
        trial_ripple_rates_df = _empty_df(TRIAL_RATE_COLUMNS)

    electrode_ripple_rates_df = pd.concat(
        [result["electrode_ripple_rates_df"] for result in subject_results_list if not result["electrode_ripple_rates_df"].empty],
        ignore_index=True,
    ) if subject_results_list else _empty_df(ELECTRODE_RATE_COLUMNS)
    if electrode_ripple_rates_df.empty:
        electrode_ripple_rates_df = _empty_df(ELECTRODE_RATE_COLUMNS)

    qc_parts = [discovery_qc_df]
    qc_parts.extend(result["qc_df"] for result in subject_results_list if not result["qc_df"].empty)
    batch_qc_df = pd.concat(qc_parts, ignore_index=True) if qc_parts else _empty_df(QC_COLUMNS)

    return {
        "subject_ids": subject_ids,
        "discovered_subject_ids": discovered_subjects,
        "behavior_subject_sets": behavior_subject_sets,
        "backend_requested": backend,
        "backend_used": backend_used,
        "ripple_events_df": ripple_events_df,
        "trial_ripple_rates_df": trial_ripple_rates_df,
        "electrode_ripple_rates_df": electrode_ripple_rates_df,
        "batch_qc_df": batch_qc_df,
        "subject_results": subject_results,
    }


In [ ]:
# Synthetic sanity checks for contiguous-segment merge and duration rules
assert find_contiguous_segments([0, 1, 1, 0, 1, 1, 1, 0]) == [(1, 2), (4, 6)]
assert merge_close_segments([(10, 20), (30, 40)], gap_samps=9) == [(10, 40)]
assert merge_close_segments([(10, 20), (31, 40)], gap_samps=9) == [(10, 20), (31, 40)]
assert duration_rejection_reason(length_samps=5, sfreq=500.0) == "too_short"
assert duration_rejection_reason(length_samps=150, sfreq=500.0) == "too_long"
assert duration_rejection_reason(length_samps=25, sfreq=500.0) is None
print("Synthetic sanity checks passed.")


In [ ]:
results = run_subject_ripple_pipeline(subject_id=SUBJECT_ID, make_plots=False)

ied_events_df = results["ied_events_df"]
ripple_candidates_df = results["ripple_candidates_df"]
ripple_events_df = results["ripple_events_df"]
trial_ripple_rates_df = results["trial_ripple_rates_df"]
electrode_ripple_rates_df = results["electrode_ripple_rates_df"]
qc_df = results["qc_df"]

discovered_subject_ids, discovery_qc_df, behavior_subject_sets = discover_usable_subject_ids()
feedback_behavior_subject_ids = behavior_subject_sets["feedback_start"]
cue_behavior_subject_ids = behavior_subject_sets["cue_on"]
print(f"Discovered usable subjects ({len(discovered_subject_ids)}): {discovered_subject_ids}")
print(f"Feedback behavior subjects ({len(feedback_behavior_subject_ids)}): {feedback_behavior_subject_ids}")
print(f"Cue behavior subjects ({len(cue_behavior_subject_ids)}): {cue_behavior_subject_ids}")
assert set(feedback_behavior_subject_ids).issubset(set(discovered_subject_ids))

print(f"Subject: {results['subject_id']}")
print("IED counts by event:")
display(ied_events_df.groupby(["event"]).size().rename("n_ieds").reset_index() if not ied_events_df.empty else pd.DataFrame())
print("Ripple candidate counts by event:")
display(ripple_candidates_df.groupby(["event"]).size().rename("n_candidates").reset_index() if not ripple_candidates_df.empty else pd.DataFrame())
print("Retained ripple counts by event:")
display(ripple_events_df.groupby(["event"]).size().rename("n_retained").reset_index() if not ripple_events_df.empty else pd.DataFrame())
print("Trial ripple-rate head:")
display(trial_ripple_rates_df.head())
print("Electrode ripple-rate head:")
display(electrode_ripple_rates_df.head())
print("QC rows:")
display(qc_df)

if not ripple_events_df.empty:
    assert ripple_events_df["ripple_duration_s"].between(RIPPLE_MIN_DURATION_S, RIPPLE_MAX_DURATION_S).all()
    for event_name, spec in EVENT_SPECS.items():
        event_df = ripple_events_df[ripple_events_df["event"] == event_name]
        if not event_df.empty:
            assert event_df["ripple_start_s"].between(spec["window"][0], spec["window"][1]).all()
            assert event_df["ripple_end_s"].between(spec["window"][0], spec["window"][1]).all()
    assert (~ripple_events_df["ied_within_100ms"]).all()
    retained_counts = ripple_events_df.groupby("event").size().to_dict()
    assert retained_counts == {"cue_on": 206, "feedback_start": 134}
    print("Retained ripple assertions passed.")
else:
    print("No retained ripples were found.")

assert trial_ripple_rates_df.groupby(["participant", "event", "trial_number"]).size().eq(1).all()
assert electrode_ripple_rates_df.groupby(["participant", "event", "electrode"]).size().eq(1).all()
print("Single-subject table integrity assertions passed.")

batch_results_seq = run_ripple_pipeline_batch(subject_ids=discovered_subject_ids, n_jobs=1, backend="loky", make_plots=False)
batch_results_parallel = run_ripple_pipeline_batch(subject_ids=discovered_subject_ids, n_jobs=-1, backend="loky", make_plots=False)

batch_ripple_events_df = batch_results_parallel["ripple_events_df"]
trial_ripple_rates_batch_df = batch_results_parallel["trial_ripple_rates_df"]
electrode_ripple_rates_batch_df = batch_results_parallel["electrode_ripple_rates_df"]
batch_qc_df = batch_results_parallel["batch_qc_df"]

seq_counts = batch_results_seq["ripple_events_df"].groupby(["participant", "event"]).size().sort_index()
par_counts = batch_results_parallel["ripple_events_df"].groupby(["participant", "event"]).size().sort_index()
pd.testing.assert_series_equal(seq_counts, par_counts)
assert trial_ripple_rates_batch_df.groupby(["participant", "event", "trial_number"]).size().eq(1).all()
assert electrode_ripple_rates_batch_df.groupby(["participant", "event", "electrode"]).size().eq(1).all()
feedback_trial_df = trial_ripple_rates_batch_df[trial_ripple_rates_batch_df["event"] == "feedback_start"].copy()
cue_trial_df = trial_ripple_rates_batch_df[trial_ripple_rates_batch_df["event"] == "cue_on"].copy()
assert set(feedback_trial_df["participant"].unique()) == set(feedback_behavior_subject_ids)
assert set(cue_trial_df["participant"].unique()) == set(cue_behavior_subject_ids)
assert len(feedback_trial_df) == (len(feedback_behavior_subject_ids) * 40)
assert len(cue_trial_df) == (len(cue_behavior_subject_ids) * 80)
print("Batch parallel safety and table integrity assertions passed.")

print("Batch retained ripple counts by participant/event:")
display(batch_ripple_events_df.groupby(["participant", "event"]).size().rename("n_retained").reset_index().head(20))
print("Batch trial ripple-rate head:")
display(trial_ripple_rates_batch_df.head())
print("Batch electrode ripple-rate head:")
display(electrode_ripple_rates_batch_df.head())
print("Batch QC head:")
display(batch_qc_df.head(20))


In [ ]:
trial_ripple_rates_batch_df.keys()

In [ ]:
trial_ripple_rates_batch_df[trial_ripple_rates_batch_df.event=='feedback_start']

In [ ]:
trial_ripple_rates_batch_df = trial_ripple_rates_batch_df.rename(columns={'trial_number':'trials'})

In [ ]:
test_df = trial_ripple_rates_batch_df[trial_ripple_rates_batch_df.event=='feedback_start'].merge(learn_df, 
on=['participant', 'trials'])

In [ ]:
test_df

In [ ]:
sns.barplot(data=test_df,
x='rpe_cat', y='n_ripples', errorbar='se')

In [ ]:
from membandit.granger import _normalize_rpe_cat_series
from membandit.behavior import load_behavior_tables, preprocess_combined_df, preprocess_learn_df

learn_df_raw, combined_df_raw = load_behavior_tables(str(LOAD_DIR))
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

In [ ]:
# Ripple behavior: simple event-specific merges (learn_df for feedback, combined_df for cue)
from membandit.behavior import load_behavior_tables, preprocess_combined_df, preprocess_learn_df

learn_df_raw, combined_df_raw = load_behavior_tables(str(LOAD_DIR))
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

if "condition" in learn_df.columns:
    learn_df = learn_df[learn_df["condition"] == CONDITION].copy()
if "condition" in combined_df.columns:
    combined_df = combined_df[combined_df["condition"] == CONDITION].copy()

ripple_trial_df = trial_ripple_rates_batch_df.copy()
# If an earlier scratch cell renamed trial_number -> trials on the whole batch, restore the standard name.
if "trial_number" not in ripple_trial_df.columns and "trials" in ripple_trial_df.columns:
    ripple_trial_df = ripple_trial_df.rename(columns={"trials": "trial_number"})

# feedback_start: trial_number matches learn_df["trials"]
_fb = ripple_trial_df[ripple_trial_df["event"] == "feedback_start"].copy()
_fb = _fb.rename(columns={"trial_number": "trials"})
feedback_df = _fb.merge(learn_df, on=["participant", "trials"], how="left")

# cue_on: trial_number matches combined_df["trials_mem"]
_cue = ripple_trial_df[ripple_trial_df["event"] == "cue_on"].copy()
_cue = _cue.rename(columns={"trial_number": "trials_mem"})
cue_df = _cue.merge(combined_df, on=["participant", "trials_mem"], how="left")

print("feedback_df shape:", feedback_df.shape)
if "trials" in feedback_df.columns:
    print("  trials null:", int(feedback_df["trials"].isna().sum()))
if "rpe_cat" in feedback_df.columns:
    print("  rpe_cat null:", int(feedback_df["rpe_cat"].isna().sum()))

print("cue_df shape:", cue_df.shape)
if "trials_mem" in cue_df.columns:
    print("  trials_mem null:", int(cue_df["trials_mem"].isna().sum()))
if "memory_outcome" in cue_df.columns:
    print("  memory_outcome null:", int(cue_df["memory_outcome"].isna().sum()))

display(feedback_df.head())
display(cue_df.head())

ripple_ev = batch_ripple_events_df.copy()


def _event_window_s(event_name):
    w = EVENT_SPECS[event_name]["window"]
    return float(w[1] - w[0])


# Per-trial per-electrode: full grid (all trial x electrode pairs with 0 ripples where none detected)
electrode_rates_batch = electrode_ripple_rates_batch_df.copy()


def _full_electrode_trial_table(event_name, trial_rename_to, behavior_df):
    """Cross join analyzed trials x HPC electrodes for this event, then attach ripple counts (0 if none)."""
    trials_ev = ripple_trial_df[ripple_trial_df["event"] == event_name].copy()
    if trials_ev.empty:
        return pd.DataFrame()
    trial_keys = trials_ev[
        ["participant", "trial_number", "condition", "trial_column", "trial_window_s", "n_hpc_electrodes"]
    ].drop_duplicates(subset=["participant", "trial_number"])

    elec_ev = electrode_rates_batch[electrode_rates_batch["event"] == event_name][
        ["participant", "electrode", "electrode_region", "electrode_hemi"]
    ].drop_duplicates()
    if elec_ev.empty:
        return pd.DataFrame()

    grid = trial_keys.merge(elec_ev, on="participant", how="inner")
    tw = float(_event_window_s(event_name))
    if "trial_window_s" in grid.columns:
        grid["trial_window_s"] = grid["trial_window_s"].fillna(tw)
    else:
        grid["trial_window_s"] = tw

    if ripple_ev.empty:
        grid["n_ripples"] = 0
    else:
        cnt = (
            ripple_ev[ripple_ev["event"] == event_name]
            .groupby(["participant", "trial_number", "electrode"], as_index=False)
            .agg(n_ripples=("trial_number", "count"))
        )
        grid = grid.merge(cnt, on=["participant", "trial_number", "electrode"], how="left")
        grid["n_ripples"] = pd.to_numeric(grid["n_ripples"], errors="coerce").fillna(0).astype(int)

    grid["ripple_rate_hz"] = grid["n_ripples"] / grid["trial_window_s"]
    grid["event"] = event_name
    out = grid.rename(columns={"trial_number": trial_rename_to})
    return out.merge(behavior_df, on=["participant", trial_rename_to], how="left")


if electrode_rates_batch.empty:
    print("electrode_ripple_rates_batch_df is empty; skipping electrode grids.")
    feedback_electrode_df = pd.DataFrame()
    cue_electrode_df = pd.DataFrame()
else:
    feedback_electrode_df = _full_electrode_trial_table(
        "feedback_start", "trials", learn_df
    )
    cue_electrode_df = _full_electrode_trial_table(
        "cue_on", "trials_mem", combined_df
    )
    print("feedback_electrode_df shape:", feedback_electrode_df.shape)
    print("cue_electrode_df shape:", cue_electrode_df.shape)
    display(feedback_electrode_df.head())
    display(cue_electrode_df.head())





In [ ]:
_fb

In [ ]:
learn_df.trials

In [ ]:
plot_results = run_subject_ripple_pipeline(subject_id=SUBJECT_ID, make_plots=True)
example_fig = make_ripple_example_panels(plot_results, n_examples=TOP_N_EXAMPLES, context_s=EXAMPLE_CONTEXT_S)
summary_figures = plot_results["summary_figures"]
assert set(summary_figures.keys()) == {event_name for event_name in EVENT_SPECS if not plot_results['ripple_events_df'][plot_results['ripple_events_df']['event'] == event_name].empty}


In [ ]:
# Grand average: all hippocampal electrodes, all events (ripple-weighted: equal weight per ripple)
from pathlib import Path
from scipy.stats import sem

GRAND_AVG_PLOT_DIR = Path("/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots")
GRAND_AVG_PLOT_DIR.mkdir(parents=True, exist_ok=True)

if "plot_results" not in globals() or plot_results is None:
    raise ValueError("Run the previous cell first to set plot_results.")
if not plot_results.get("event_payloads"):
    raise ValueError("plot_results must include event_payloads; re-run with make_plots=True.")
if plot_results["ripple_events_df"].empty:
    print("No ripples in ripple_events_df; skipping grand-average plot.")
else:
    grouped = _collect_locked_snippets(plot_results, groupby_event=False)
    if not grouped or "pooled" not in grouped:
        print("No retained ripples with full plotting context for pooled grand average.")
    else:
        p = grouped["pooled"]
        plot_time_axis = p["plot_time_axis"]
        plot_mask = p["plot_mask"]
        raw_uv = p["raw_compute"][:, plot_mask] * UV_SCALE
        rip_uv = p["ripple_compute"][:, plot_mask] * UV_SCALE
        meta = p["metadata"]
        n_ripples = int(raw_uv.shape[0])
        n_electrodes = int(meta["electrode"].nunique())
        sid = plot_results["subject_id"]

        mean_raw = np.mean(raw_uv, axis=0)
        sem_raw = sem(raw_uv, axis=0)
        mean_rip = np.mean(rip_uv, axis=0)
        sem_rip = sem(rip_uv, axis=0)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
        configs = [
            (axes[0], mean_raw, sem_raw, "black", "Raw LFP"),
            (axes[1], mean_rip, sem_rip, "dimgray", "70–180 Hz trace"),
        ]
        for ax, m, s, color, ttl in configs:
            ax.fill_between(plot_time_axis, m - s, m + s, color="0.78", alpha=0.85, linewidth=0)
            ax.plot(plot_time_axis, m, color=color, linewidth=2)
            ax.axvline(0.0, color="tab:red", linestyle="--", linewidth=1.0)
            ax.set_xlabel("Time from ripple peak (s)")
            ax.set_ylabel("Amplitude (uV)")
            ax.set_title(f"{ttl}  |  mean ± SEM  |  n={n_ripples} ripples")

        fig.suptitle(
            f"{sid}  |  Grand average across all hippocampal electrodes (ripple-weighted; "
            f"{n_electrodes} contacts)",
            y=1.02,
        )
        fig.tight_layout()
        wave_path = GRAND_AVG_PLOT_DIR / f"{sid}_ripple_grand_average_waveforms.pdf"
        fig.savefig(wave_path, dpi=300, bbox_inches="tight")
        print(f"Saved {wave_path}")
        plt.show()

        tfr_payload = _compute_baseline_z_tfr(
            p["raw_compute"],
            sfreq=p["sfreq"],
            compute_time_axis=p["compute_time_axis"],
            plot_mask=plot_mask,
            plot_window_s=SUMMARY_WINDOW_S,
            baseline_window=TFR_BASELINE_WINDOW,
        )
        pooled_summary = _summarize_tfr(tfr_payload["z_power"], summary_stat="median")
        vmin, vmax = _compute_event_color_limits(pooled_summary, [])

        fig_spec, ax_spec = plt.subplots(figsize=(8, 6))
        mesh = ax_spec.pcolormesh(
            tfr_payload["plot_time_axis"],
            tfr_payload["freqs"],
            pooled_summary,
            shading="auto",
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
        )
        ax_spec.axvline(0.0, color="white", linestyle="--", linewidth=1.0)
        ax_spec.set_title(
            f"{sid}  |  Grand average ripple-locked spectrogram (all HPC; n={n_ripples}; {n_electrodes} contacts)"
        )
        ax_spec.set_xlabel("Time from ripple peak (s)")
        ax_spec.set_ylabel("Frequency (Hz)")
        cbar = fig_spec.colorbar(mesh, ax=ax_spec)
        cbar.set_label("z")
        fig_spec.tight_layout()
        spec_path = GRAND_AVG_PLOT_DIR / f"{sid}_ripple_grand_average_spectrogram.pdf"
        fig_spec.savefig(spec_path, dpi=300, bbox_inches="tight")
        print(f"Saved {spec_path}")
        plt.show()


In [ ]:
# Subject-wise ripple rate by behavior category (batch): bar = mean across subjects ± SEM; points = one subject mean per category
from membandit.granger import MEMORY_OUTCOME_CONDITIONS, _normalize_rpe_cat_series

RIPPLE_RATE_BY_CAT_DIR = Path("/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots")
RIPPLE_RATE_BY_CAT_DIR.mkdir(parents=True, exist_ok=True)

# Canonical order after normalizing learn_df vs combined_df RPE labels (learn uses negative/neutral/positive; combined uses neg/neutral/pos).
RPE_CAT_ORDER = ["neg", "neutral", "pos"]
MEMORY_OUTCOME_ORDER = list(MEMORY_OUTCOME_CONDITIONS)


def _plot_subject_category_ripple_rates(
    trial_df,
    category_col,
    category_order,
    title,
    save_path,
    *,
    normalize_rpe_cat: bool = False,
):
    required = {"participant", category_col, "ripple_rate_hz"}
    missing = required - set(trial_df.columns)
    if missing:
        print(f"Skipping plot ({title}): missing columns {sorted(missing)}")
        return
    sub = trial_df.dropna(subset=[category_col]).copy()
    if sub.empty:
        print(f"Skipping plot ({title}): no rows after dropping NA in {category_col}")
        return
    if normalize_rpe_cat:
        sub[category_col] = _normalize_rpe_cat_series(sub[category_col])
        sub = sub.dropna(subset=[category_col])
        if sub.empty:
            print(f"Skipping plot ({title}): no rows after RPE category normalization")
            return
    plot_df = (
        sub.groupby(["participant", category_col], as_index=False)["ripple_rate_hz"]
        .mean()
    )
    present = set(plot_df[category_col].astype(str).unique())
    order = [c for c in category_order if str(c) in present]
    if not order:
        order = sorted(present)

    n_subj = int(plot_df["participant"].nunique())
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=plot_df,
        x=category_col,
        y="ripple_rate_hz",
        order=order,
        errorbar="se",
        color="0.78",
        edgecolor="0.2",
        linewidth=1.0,
        ax=ax,
    )
    sns.stripplot(
        data=plot_df,
        x=category_col,
        y="ripple_rate_hz",
        order=order,
        color="0.15",
        alpha=0.85,
        size=7,
        jitter=0.12,
        linewidth=0.5,
        edgecolor="0.15",
        ax=ax,
    )
    ax.set_title(f"{title} (n subjects = {n_subj})")
    ax.set_xlabel(category_col.replace("_", " "))
    ax.set_ylabel("Mean ripple rate (Hz)\n(trial mean per subject × category)")
    sns.despine(ax=ax, top=True, right=True)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved {save_path}")
    plt.show()


if "feedback_df" not in globals() or "cue_df" not in globals():
    raise NameError("Run the ripple behavior merge cell first to define feedback_df and cue_df.")

_plot_subject_category_ripple_rates(
    feedback_df,
    category_col="rpe_cat",
    category_order=RPE_CAT_ORDER,
    title="HPC ripple rate by RPE category | feedback_start",
    save_path=RIPPLE_RATE_BY_CAT_DIR / "ripple_rate_by_rpe_cat_feedback_start.pdf",
    normalize_rpe_cat=True,
)

_plot_subject_category_ripple_rates(
    cue_df,
    category_col="rpe_cat",
    category_order=RPE_CAT_ORDER,
    title="HPC ripple rate by RPE category | cue_on",
    save_path=RIPPLE_RATE_BY_CAT_DIR / "ripple_rate_by_rpe_cat_cue_on.pdf",
    normalize_rpe_cat=True,
)

_plot_subject_category_ripple_rates(
    cue_df,
    category_col="memory_outcome",
    category_order=MEMORY_OUTCOME_ORDER,
    title="HPC ripple rate by memory outcome | cue_on",
    save_path=RIPPLE_RATE_BY_CAT_DIR / "ripple_rate_by_memory_outcome_cue_on.pdf",
)
